# PPM Benchmark Library - Full Genie Semantic Layer

This is the single comprehensive notebook for the Databricks Genie space. It contains:

1. The full Unity Catalog Metric View DDL.
2. Dimension metadata used by the Tableau dashboard.
3. Complete SQL metric definitions, including LOD-safe variants.
4. Core dashboard metric definitions across Overall, LOB/customer segment/product/claim/Blues, DP, MR, Medical Policy, and Topic.
5. Runnable SQL query samples for every dashboard grain.

Source workbook studied: `/Users/poojakabadi/Desktop/test.twb`.
Source draft notebook studied: `/Users/poojakabadi/Desktop/Metric View.ipynb`.


## How To Use This Notebook

Run the cells in order inside Databricks:

1. Create or replace the metric view.
2. Review the dimension and metric catalogs.
3. Run validation queries for Overall, DP, MR, Medical Policy, and Topic.
4. Add the metric view to Genie and paste the LOD selection guide into Genie instructions.


## Create Metric View

This cell is the semantic layer object that Genie should use. It includes all dimensions, all metric definitions, and source-level LOD-safe denominator fields.


In [ ]:
%sql
CREATE OR REPLACE VIEW oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: PPM Benchmark Library semantic layer reverse-engineered from Tableau workbook test.twb. Includes dashboard dimensions,
  Tableau-derived calculated fields, solution-aware measures, and LOD-safe denominators for Genie Text-to-SQL.
source: |-
  WITH scoped AS (
    SELECT *
    FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
    WHERE customer IS NOT NULL
      AND customer <> 'Lib'
  ), modeled AS (
    SELECT
      scoped.*,
      CASE
        WHEN cv_rule_10 = '-1' THEN 'CV'
        WHEN cv_rule_10 = '0' THEN 'PPM'
        WHEN solution IN ('CPR','FWA') THEN solution
        ELSE solution
      END AS tableau_solution_family,
      CASE WHEN blues_indicator = 'Y' THEN 'Blues' WHEN blues_indicator = 'N' THEN 'Non-Blues' ELSE 'Unknown' END AS blues_indicator_label,
      CASE WHEN idn_yn = 'Y' THEN 'IDN' WHEN idn_yn = 'N' THEN 'Non-IDN' ELSE 'Unknown' END AS idn_indicator_label,
      CASE WHEN library_status_key = '1' THEN 'Library Rule' WHEN library_status_key = '2' THEN 'Custom Rule' ELSE 'Unknown' END AS rule_type_label,
      CASE WHEN COALESCE(disabled,'N') = 'Y' THEN 'Disabled' WHEN COALESCE(deactivated,'N') = 'Y' THEN 'Deactivated' ELSE 'Active' END AS rule_status_label,
      CASE WHEN medical_policy IS NULL THEN 'ZZ.Null' ELSE medical_policy END AS medical_policy_label,
      CASE WHEN topic IS NULL THEN 'ZZ.Null' ELSE topic END AS topic_label,
      CASE WHEN mid_rule_key IS NULL THEN 'ZZ.Null' ELSE mid_rule_key END AS mid_rule_key_label,
      CASE WHEN cotiviti_edit_position IN ('1','2','3','4','5') THEN cotiviti_edit_position ELSE 'Unknown' END AS cotiviti_edit_position_label,
      CASE WHEN primary_3rd_party_editor IS NULL OR primary_3rd_party_editor IN ('','N/A','n/a') THEN 'Unknown' ELSE primary_3rd_party_editor END AS primary_editor_label,
      CASE WHEN claims_platform IS NULL OR claims_platform IN ('','N/A','n/a') THEN 'Unknown' ELSE claims_platform END AS claims_platform_label,
      CASE
        WHEN dp_start_date IS NULL THEN 'Custom DP'
        WHEN FLOOR(months_between(current_date(), DATE(dp_start_date)) / 12) < 1 THEN 'Less than 1 Year'
        ELSE CAST(FLOOR(months_between(current_date(), DATE(dp_start_date)) / 12) AS STRING)
      END AS dp_age_bucket,
      FLOOR(months_between(current_date(), DATE(dp_start_date)) / 12) AS dp_age_years,
      FLOOR(months_between(current_date(), DATE(min_invoice_date)) / 12) AS payer_age_years,
      CASE WHEN status = 'Current' THEN 'Y' ELSE 'N' END AS current_payer_flag,
      CASE WHEN prim_core_enhanced_key IS NULL THEN 'Unclassified' ELSE CAST(prim_core_enhanced_key AS STRING) END AS core_enhanced_label,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state ORDER BY dp_key, mid_rule_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_customer_payer_state,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, insurance_desc ORDER BY dp_key, mid_rule_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_lob,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, product_type ORDER BY dp_key, mid_rule_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_product_type,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, claim_type_desc_oi ORDER BY dp_key, mid_rule_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_claim_type,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, customer_segment ORDER BY dp_key, mid_rule_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_customer_segment,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, blues_indicator ORDER BY dp_key, mid_rule_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_blues,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, dp_key ORDER BY mid_rule_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_dp,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, mid_rule_key ORDER BY dp_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_mid_rule,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, medical_policy ORDER BY dp_key, mid_rule_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_medical_policy,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, topic ORDER BY dp_key, mid_rule_key, sub_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_topic,
      CASE WHEN ROW_NUMBER() OVER (PARTITION BY customer, payer_short, final_state, sub_rule_key ORDER BY dp_key, mid_rule_key) = 1 THEN COALESCE(paid,0) ELSE 0 END AS paid_once_sub_rule
    FROM scoped
  ), totals AS (
    SELECT
      COUNT(DISTINCT customer) AS total_customers_universe,
      COUNT(DISTINCT payer_short) AS total_payers_universe,
      COUNT(DISTINCT super_payer_short) AS total_super_payers_universe,
      COUNT(DISTINCT dp_key) AS total_dps_universe,
      COUNT(DISTINCT mid_rule_key) AS total_mrs_universe,
      SUM(COALESCE(gpv,0)) AS total_gpv_universe,
      SUM(COALESCE(npv,0)) AS total_npv_universe,
      SUM(COALESCE(paid,0)) AS total_paid_raw_universe,
      SUM(paid_once_customer_payer_state) AS total_paid_dedup_universe
    FROM modeled
  )
  SELECT modeled.*, totals.*
  FROM modeled CROSS JOIN totals
dimensions:
- name: Overall
  expr: '''Overall'''
  display_name: Overall
  comment: Constant dimension used to reproduce Tableau Overall level.
- name: Customer
  expr: customer
  display_name: Customer
  comment: Customer/account from the PPM benchmark library.
  synonyms:
  - account
  - client
  - payer customer
- name: Customer Segment
  expr: customer_segment
  display_name: Customer Segment
  comment: Customer segment used in the dashboard Level 1/Level 2 selector.
  synonyms:
  - segment
  - market segment
- name: Line of Business
  expr: insurance_desc
  display_name: Line of Business
  comment: LOB / insurance line, matching Tableau LOB.
  synonyms:
  - LOB
  - insurance
  - coverage line
- name: Product Type
  expr: product_type
  display_name: Product Type
  comment: Product type slicer.
  synonyms:
  - product
  - product category
- name: Claim Type
  expr: claim_type_desc_oi
  display_name: Claim Type
  comment: Operational claim type used by the benchmark dashboard.
  synonyms:
  - claim type OI
  - claim category
- name: Claim Type Base
  expr: claim_type_desc
  display_name: Claim Type Base
  comment: Base claim type description.
  synonyms:
  - claim type description
- name: Final State
  expr: final_state
  display_name: Final State
  comment: Final payer/customer state, used by the Tableau State parameter.
  synonyms:
  - state
  - payer state
  - geography
- name: Discrete State
  expr: discrete_state
  display_name: Discrete State
  comment: Discrete state field from the source.
- name: Payer
  expr: payer_short
  display_name: Payer
  comment: Payer short name.
  synonyms:
  - payer short
  - health plan
- name: Payer Full Name
  expr: payer_name
  display_name: Payer Full Name
  comment: Full payer name.
  synonyms:
  - payer name
- name: Super Payer
  expr: super_payer_short
  display_name: Super Payer
  comment: Super payer short name.
  synonyms:
  - parent payer
  - super payer short
- name: Super Payer Full Name
  expr: super_payer_name
  display_name: Super Payer Full Name
  comment: Full super payer name.
- name: Blues Indicator
  expr: blues_indicator_label
  display_name: Blues Indicator
  comment: Blues vs Non-Blues label derived from blues_indicator.
  synonyms:
  - BCBS
  - Blue Cross
  - Blue Shield
  - Blues
- name: Blues Raw Flag
  expr: blues_indicator
  display_name: Blues Raw Flag
  comment: Raw Y/N Blues flag.
- name: Blues Home Host
  expr: blues_home_host
  display_name: Blues Home Host
  comment: Blues home/host indicator from source.
- name: IDN Indicator
  expr: idn_indicator_label
  display_name: IDN Indicator
  comment: IDN vs Non-IDN label derived from idn_yn.
  synonyms:
  - IDN
- name: Status
  expr: status
  display_name: Status
  comment: Payer lifecycle status, for current/retired filtering.
- name: Current Payer Flag
  expr: current_payer_flag
  display_name: Current Payer Flag
  comment: Y when payer status is Current; retained to mimic active payer dashboard filter.
- name: Rule Status
  expr: rule_status_label
  display_name: Rule Status
  comment: Active, Disabled, or Deactivated rule status.
  synonyms:
  - active rules
  - disabled rules
  - deactivated rules
- name: Disabled
  expr: disabled
  display_name: Disabled
  comment: Raw disabled flag.
- name: Deactivated
  expr: deactivated
  display_name: Deactivated
  comment: Raw deactivated flag.
- name: Rule Type
  expr: rule_type_label
  display_name: Rule Type
  comment: Library Rule vs Custom Rule, matching Tableau rule type filter.
  synonyms:
  - library rule
  - custom rule
- name: Solution Family
  expr: tableau_solution_family
  display_name: Solution Family
  comment: Tableau-compatible solution family derived from cv_rule_10 and solution.
  synonyms:
  - solution
  - PPM
  - CV
  - FWA
  - CPR
- name: Source Solution
  expr: solution
  display_name: Source Solution
  comment: Raw source solution value.
- name: CV Rule 10
  expr: cv_rule_10
  display_name: CV Rule 10
  comment: CV rule 10 flag used by Tableau solution logic.
- name: Core Enhanced
  expr: core_enhanced_label
  display_name: Core Enhanced
  comment: Core/Enhanced key/classification from Tableau source; use values available in prim_core_enhanced_key.
  synonyms:
  - core
  - enhanced
- name: Policy Collection Bundle
  expr: pcbi_master_list
  display_name: Policy Collection Bundle
  comment: Policy collection/bundle list used by Tableau Policy Collection/Bundle parameter.
  synonyms:
  - collection
  - bundle
  - policy bundle
- name: Policy Bundle Indicator
  expr: policy_bundle_indicator
  display_name: Policy Bundle Indicator
  comment: Policy bundle indicator.
- name: Policy Package Indicator
  expr: policy_package_indicator
  display_name: Policy Package Indicator
  comment: Policy package indicator.
- name: Policy Package Individual Indicator
  expr: policy_package_individual_indicator
  display_name: Policy Package Individual Indicator
  comment: Policy package individual indicator.
- name: Decision Point
  expr: dp_key
  display_name: Decision Point
  comment: Decision Point key, the main DP dashboard grain.
  synonyms:
  - DP
  - rule
  - decision point key
- name: Decision Point Description
  expr: dp_desc
  display_name: Decision Point Description
  comment: Decision Point description.
  synonyms:
  - DP description
  - rule description
- name: DP Age Bucket
  expr: dp_age_bucket
  display_name: DP Age Bucket
  comment: 'Tableau DP Age in years bucket: Custom DP, Less than 1 Year, or whole years.'
  synonyms:
  - DP age
  - age in years
- name: DP Age Years
  expr: dp_age_years
  display_name: DP Age Years
  comment: Numeric DP age in years based on dp_start_date.
- name: DP Start Date
  expr: DATE(dp_start_date)
  display_name: DP Start Date
  comment: Decision Point start date.
- name: Medical Policy
  expr: medical_policy_label
  display_name: Medical Policy
  comment: 'Medical policy with Tableau null handling: null becomes ZZ.Null.'
  synonyms:
  - policy
  - medical policy
- name: Latest Medical Policy
  expr: latest_medical1_policy
  display_name: Latest Medical Policy
  comment: Latest medical policy field.
- name: Topic
  expr: topic_label
  display_name: Topic
  comment: 'Topic with Tableau null handling: null becomes ZZ.Null.'
  synonyms:
  - clinical topic
  - category
- name: Mid Rule Key
  expr: mid_rule_key_label
  display_name: Mid Rule Key
  comment: Mid Rule key with Tableau null handling.
  synonyms:
  - MR
  - MR key
  - mid rule
- name: Mid Rule Version
  expr: mid_rule_version
  display_name: Mid Rule Version
  comment: Mid Rule version.
- name: Mid Rule Dot Version
  expr: mid_rule_dot_version
  display_name: Mid Rule Dot Version
  comment: Mid Rule dot version.
- name: Sub Rule Key
  expr: sub_rule_key
  display_name: Sub Rule Key
  comment: Sub-rule key.
  synonyms:
  - sub rule
  - subrule
- name: Sub Rule Description
  expr: sub_rule_desc
  display_name: Sub Rule Description
  comment: Sub-rule description.
- name: DP for Sub Rule
  expr: dp_key_4_sub_rule_key
  display_name: DP for Sub Rule
  comment: DP key associated to sub-rule key.
- name: DP Description for Sub Rule
  expr: dp_desc_4_sub_rule_key
  display_name: DP Description for Sub Rule
  comment: DP description associated to sub-rule key.
- name: Topic Key for Sub Rule
  expr: topic_key_4_sub_rule_key
  display_name: Topic Key for Sub Rule
  comment: Topic key associated to sub-rule key.
- name: Topic for Sub Rule
  expr: topic_title_4_sub_rule_key
  display_name: Topic for Sub Rule
  comment: Topic title associated to sub-rule key.
- name: Medical Policy Key for Sub Rule
  expr: med_pol_key_4_sub_rule_key
  display_name: Medical Policy Key for Sub Rule
  comment: Medical policy key associated to sub-rule key.
- name: Medical Policy for Sub Rule
  expr: med_pol_title_4_sub_rule_key
  display_name: Medical Policy for Sub Rule
  comment: Medical policy title associated to sub-rule key.
- name: Policy Type Key for Sub Rule
  expr: pol_type_key_4_sub_rule_key
  display_name: Policy Type Key for Sub Rule
  comment: Policy type key associated to sub-rule key.
- name: Policy Type for Sub Rule
  expr: pol_type_desc_4_sub_rule_key
  display_name: Policy Type for Sub Rule
  comment: Policy type description associated to sub-rule key.
- name: Claims Platform
  expr: claims_platform_label
  display_name: Claims Platform
  comment: Claims platform with Tableau Unknown handling.
  synonyms:
  - platform
- name: Cotiviti Edit Position
  expr: cotiviti_edit_position_label
  display_name: Cotiviti Edit Position
  comment: Cotiviti edit position normalized to 1-5 or Unknown.
  synonyms:
  - edit position
  - processing position
- name: Primary 3rd Party Editor
  expr: primary_editor_label
  display_name: Primary 3rd Party Editor
  comment: Primary third-party editor with Tableau Unknown handling.
  synonyms:
  - competitor
  - primary editor
- name: Secondary 3rd Party Editor
  expr: secondary_3rd_party_editor
  display_name: Secondary 3rd Party Editor
  comment: Secondary third-party editor.
- name: Has Competitor
  expr: CASE WHEN primary_editor_label = 'Unknown' THEN 'No Competitor' ELSE 'Has Competitor' END
  display_name: Has Competitor
  comment: Whether a primary third-party editor is present.
  synonyms:
  - competitive account
- name: SVP
  expr: svp
  display_name: SVP
  comment: SVP owner dimension.
- name: CEL
  expr: cel
  display_name: CEL
  comment: Client engagement lead dimension.
- name: CSD
  expr: csd
  display_name: CSD
  comment: CSD owner dimension.
- name: CSM
  expr: csm
  display_name: CSM
  comment: CSM owner dimension.
- name: CPM
  expr: cpm
  display_name: CPM
  comment: CPM owner dimension.
- name: BA
  expr: ba
  display_name: BA
  comment: BA owner dimension.
- name: Client Medical Director
  expr: client_medical_director
  display_name: Client Medical Director
  comment: Client medical director.
- name: Min Invoice Date
  expr: DATE(min_invoice_date)
  display_name: Min Invoice Date
  comment: Minimum invoice date.
- name: Max Invoice Date
  expr: DATE(max_invoice_date)
  display_name: Max Invoice Date
  comment: Maximum invoice date.
- name: Payer Age Years
  expr: payer_age_years
  display_name: Payer Age Years
  comment: Payer age in years from min_invoice_date.
- name: Max Invoice Year
  expr: max_invoice_year
  display_name: Max Invoice Year
  comment: Maximum invoice year.
- name: Latest Ref Date
  expr: latest_ref_date
  display_name: Latest Ref Date
  comment: Latest data refresh/reference date.
- name: Insurance Key
  expr: insurance_key
  display_name: Insurance Key
  comment: Insurance key.
- name: Customer Code
  expr: customer_code
  display_name: Customer Code
  comment: Customer code.
- name: Payer Key
  expr: payer_key
  display_name: Payer Key
  comment: Payer key.
- name: Payer Short Code
  expr: payer_short_code
  display_name: Payer Short Code
  comment: Payer short code.
- name: Super Payer Key
  expr: super_payer_key
  display_name: Super Payer Key
  comment: Super payer key.
- name: Super Payer Code
  expr: super_payer_code
  display_name: Super Payer Code
  comment: Super payer code.
- name: CV Code
  expr: cv_code
  display_name: CV Code
  comment: CV code.
- name: CV Client
  expr: cv_client
  display_name: CV Client
  comment: CV client flag.
- name: NDC Data
  expr: ndc_data
  display_name: NDC Data
  comment: NDC data flag.
- name: Client Name
  expr: client_name
  display_name: Client Name
  comment: Client name from source.
- name: PCI Master List
  expr: pci_master_list
  display_name: PCI Master List
  comment: PCI master list.
- name: PBI Master List
  expr: pbi_master_list
  display_name: PBI Master List
  comment: PBI master list.
- name: PCBI Master List
  expr: pcbi_master_list
  display_name: PCBI Master List
  comment: Combined PCBI master list.
measures:
- name: Total Paid Raw
  expr: SUM(COALESCE(paid,0))
  display_name: Total Paid Raw
  comment: Raw sum of paid. Use mainly for Level 1 dashboard cuts where paid is not fanned out.
  synonyms:
  - paid
  - claim volume
  - gross paid
- name: Total Paid Dedup Customer Payer State
  expr: SUM(COALESCE(paid_once_customer_payer_state,0))
  display_name: Total Paid Dedup Customer Payer State
  comment: Paid counted once per customer x payer x final state. This is the safest overall paid denominator when DP/MR/policy
    fan-out is present.
- name: Total Paid Customer Segment Denominator
  expr: SUM(COALESCE(paid_once_customer_segment,0))
  display_name: Total Paid Customer Segment Denominator
  comment: Paid denominator counted once per customer x payer x final state x customer segment. Use for Overall Benchmark customer segment cuts.
- name: Total Paid LOB Denominator
  expr: SUM(COALESCE(paid_once_lob,0))
  display_name: Total Paid LOB Denominator
  comment: Paid denominator counted once per customer x payer x final state x line of business. Use for Overall Benchmark LOB cuts.
- name: Total Paid Product Type Denominator
  expr: SUM(COALESCE(paid_once_product_type,0))
  display_name: Total Paid Product Type Denominator
  comment: Paid denominator counted once per customer x payer x final state x product type. Use for Overall Benchmark product type cuts.
- name: Total Paid Claim Type Denominator
  expr: SUM(COALESCE(paid_once_claim_type,0))
  display_name: Total Paid Claim Type Denominator
  comment: Paid denominator counted once per customer x payer x final state x claim type. Use for Overall Benchmark claim type cuts.
- name: Total Paid Blues Denominator
  expr: SUM(COALESCE(paid_once_blues,0))
  display_name: Total Paid Blues Denominator
  comment: Paid denominator counted once per customer x payer x final state x Blues indicator. Use for Overall Benchmark Blues vs Non-Blues cuts.
- name: Total Paid DP Denominator
  expr: SUM(COALESCE(paid_once_dp,0))
  display_name: Total Paid DP Denominator
  comment: Paid denominator counted once per customer x payer x final state x DP. Use for DP-level percentage questions.
- name: Total Paid MR Denominator
  expr: SUM(COALESCE(paid_once_mid_rule,0))
  display_name: Total Paid MR Denominator
  comment: Paid denominator counted once per customer x payer x final state x MR. Use for MR-level percentage questions.
- name: Total Paid Medical Policy Denominator
  expr: SUM(COALESCE(paid_once_medical_policy,0))
  display_name: Total Paid Medical Policy Denominator
  comment: Paid denominator counted once per customer x payer x final state x medical policy. Use for policy-level percentage
    questions.
- name: Total Paid Topic Denominator
  expr: SUM(COALESCE(paid_once_topic,0))
  display_name: Total Paid Topic Denominator
  comment: Paid denominator counted once per customer x payer x final state x topic. Use for topic-level percentage questions.
- name: Total Paid Sub Rule Denominator
  expr: SUM(COALESCE(paid_once_sub_rule,0))
  display_name: Total Paid Sub Rule Denominator
  comment: Paid denominator counted once per customer x payer x final state x sub-rule. Use for sub-rule-level percentage
    questions.
- name: Total GPV
  expr: SUM(COALESCE(gpv,0))
  display_name: Total GPV
  comment: Gross payment value / gross savings.
  synonyms:
  - GPV
  - gross savings
  - gross payment value
- name: Total Adjustments
  expr: SUM(COALESCE(adjustments,0))
  display_name: Total Adjustments
  comment: Adjustment dollars applied against GPV.
  synonyms:
  - adjustments
  - APV dollars
- name: Total NPV
  expr: SUM(COALESCE(npv,0))
  display_name: Total NPV
  comment: Net payment value / net savings.
  synonyms:
  - NPV
  - net savings
  - net payment value
- name: Total EPV
  expr: SUM(COALESCE(epv,0))
  display_name: Total EPV
  comment: Expected payment value from source.
  synonyms:
  - EPV
- name: Total Unapplied
  expr: SUM(COALESCE(unapplied,0))
  display_name: Total Unapplied
  comment: Unapplied savings opportunity.
  synonyms:
  - unapplied
  - missed opportunity
- name: GPV Percent Raw
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid,0)),0)
  display_name: GPV Percent Raw
  comment: Raw GPV percent. Correct only when paid is not duplicated by the selected dimensions.
  synonyms:
  - GPV percent
  - GPV %
- name: NPV Percent Raw
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid,0)),0)
  display_name: NPV Percent Raw
  comment: Raw NPV percent. Correct only when paid is not duplicated by the selected dimensions.
  synonyms:
  - NPV percent
  - NPV %
- name: GPV Percent Overall LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_payer_state,0)),0)
  display_name: GPV Percent Overall LOD
  comment: LOD-safe GPV percent using customer-payer-state paid denominator.
  synonyms:
  - LOD GPV percent
  - safe GPV percent
- name: NPV Percent Overall LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_payer_state,0)),0)
  display_name: NPV Percent Overall LOD
  comment: LOD-safe NPV percent using customer-payer-state paid denominator.
  synonyms:
  - LOD NPV percent
  - safe NPV percent
- name: GPV Percent by Customer Segment LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_segment,0)),0)
  display_name: GPV Percent by Customer Segment LOD
  comment: Use when grouping/filtering by Customer Segment in the Overall Benchmark.
  synonyms:
  - customer segment GPV percent
- name: NPV Percent by Customer Segment LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_segment,0)),0)
  display_name: NPV Percent by Customer Segment LOD
  comment: Use when grouping/filtering by Customer Segment in the Overall Benchmark.
  synonyms:
  - customer segment NPV percent
- name: GPV Percent by LOB LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_lob,0)),0)
  display_name: GPV Percent by LOB LOD
  comment: Use when grouping/filtering by Line of Business in the Overall Benchmark.
  synonyms:
  - LOB GPV percent
  - line of business GPV percent
- name: NPV Percent by LOB LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_lob,0)),0)
  display_name: NPV Percent by LOB LOD
  comment: Use when grouping/filtering by Line of Business in the Overall Benchmark.
  synonyms:
  - LOB NPV percent
  - line of business NPV percent
- name: GPV Percent by Product Type LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_product_type,0)),0)
  display_name: GPV Percent by Product Type LOD
  comment: Use when grouping/filtering by Product Type in the Overall Benchmark.
  synonyms:
  - product GPV percent
- name: NPV Percent by Product Type LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_product_type,0)),0)
  display_name: NPV Percent by Product Type LOD
  comment: Use when grouping/filtering by Product Type in the Overall Benchmark.
  synonyms:
  - product NPV percent
- name: GPV Percent by Claim Type LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_claim_type,0)),0)
  display_name: GPV Percent by Claim Type LOD
  comment: Use when grouping/filtering by Claim Type in the Overall Benchmark.
  synonyms:
  - claim type GPV percent
- name: NPV Percent by Claim Type LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_claim_type,0)),0)
  display_name: NPV Percent by Claim Type LOD
  comment: Use when grouping/filtering by Claim Type in the Overall Benchmark.
  synonyms:
  - claim type NPV percent
- name: GPV Percent by Blues LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_blues,0)),0)
  display_name: GPV Percent by Blues LOD
  comment: Use when grouping/filtering by Blues Indicator in the Overall Benchmark.
  synonyms:
  - Blues GPV percent
  - BCBS GPV percent
- name: NPV Percent by Blues LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_blues,0)),0)
  display_name: NPV Percent by Blues LOD
  comment: Use when grouping/filtering by Blues Indicator in the Overall Benchmark.
  synonyms:
  - Blues NPV percent
  - BCBS NPV percent
- name: GPV Percent by DP LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_dp,0)),0)
  display_name: GPV Percent by DP LOD
  comment: Use when grouping/filtering by Decision Point.
  synonyms:
  - DP GPV percent
  - GPV percent by rule
- name: NPV Percent by DP LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_dp,0)),0)
  display_name: NPV Percent by DP LOD
  comment: Use when grouping/filtering by Decision Point.
  synonyms:
  - DP NPV percent
  - NPV percent by rule
- name: GPV Percent by MR LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_mid_rule,0)),0)
  display_name: GPV Percent by MR LOD
  comment: Use when grouping/filtering by Mid Rule.
  synonyms:
  - MR GPV percent
- name: NPV Percent by MR LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_mid_rule,0)),0)
  display_name: NPV Percent by MR LOD
  comment: Use when grouping/filtering by Mid Rule.
  synonyms:
  - MR NPV percent
- name: GPV Percent by Medical Policy LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_medical_policy,0)),0)
  display_name: GPV Percent by Medical Policy LOD
  comment: Use when grouping/filtering by Medical Policy.
  synonyms:
  - policy GPV percent
- name: NPV Percent by Medical Policy LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_medical_policy,0)),0)
  display_name: NPV Percent by Medical Policy LOD
  comment: Use when grouping/filtering by Medical Policy.
  synonyms:
  - policy NPV percent
- name: GPV Percent by Topic LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_topic,0)),0)
  display_name: GPV Percent by Topic LOD
  comment: Use when grouping/filtering by Topic.
  synonyms:
  - topic GPV percent
- name: NPV Percent by Topic LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_topic,0)),0)
  display_name: NPV Percent by Topic LOD
  comment: Use when grouping/filtering by Topic.
  synonyms:
  - topic NPV percent
- name: GPV Percent by Sub Rule LOD
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_sub_rule,0)),0)
  display_name: GPV Percent by Sub Rule LOD
  comment: Use when grouping/filtering by Sub Rule.
  synonyms:
  - subrule GPV percent
- name: NPV Percent by Sub Rule LOD
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_sub_rule,0)),0)
  display_name: NPV Percent by Sub Rule LOD
  comment: Use when grouping/filtering by Sub Rule.
  synonyms:
  - subrule NPV percent
- name: APV Percent
  expr: 100.0 * SUM(COALESCE(adjustments,0)) / NULLIF(SUM(COALESCE(gpv,0)),0)
  display_name: APV Percent
  comment: 'Adjustment percent: adjustments divided by GPV, matching Tableau APV%.'
  synonyms:
  - APV
  - adjustment percent
- name: Unapplied Percent
  expr: 100.0 * SUM(COALESCE(unapplied,0)) / NULLIF(SUM(COALESCE(unapplied,0)) + SUM(COALESCE(gpv,0)),0)
  display_name: Unapplied Percent
  comment: Unapplied divided by unapplied plus GPV, matching Tableau Unapplied%.
  synonyms:
  - unapplied rate
  - missed opportunity percent
- name: Total Lines
  expr: SUM(COALESCE(total_lines,0))
  display_name: Total Lines
  comment: Total claim/service lines.
- name: Edited Lines
  expr: SUM(COALESCE(edited_lines,0))
  display_name: Edited Lines
  comment: Edited lines.
- name: Adjusted Lines
  expr: SUM(COALESCE(adjusted_lines,0))
  display_name: Adjusted Lines
  comment: Adjusted lines.
- name: Adjusted Lines Percent
  expr: 100.0 * SUM(COALESCE(adjusted_lines,0)) / NULLIF(SUM(COALESCE(edited_lines,0)),0)
  display_name: Adjusted Lines Percent
  comment: Adjusted lines divided by edited lines.
  synonyms:
  - adjusted line rate
- name: Edits per 1000
  expr: 1000.0 * SUM(COALESCE(edited_lines,0)) / NULLIF(SUM(COALESCE(total_lines,0)),0)
  display_name: Edits per 1000
  comment: Edited lines per 1,000 total lines.
  synonyms:
  - edits per thousand
  - edit rate
- name: Original Paid per TBA Line
  expr: SUM(COALESCE(paid,0)) / NULLIF(SUM(COALESCE(total_lines,0)),0)
  display_name: Original Paid per TBA Line
  comment: Paid dollars per total/TBA line.
- name: Original Paid per Edit
  expr: SUM(CASE WHEN COALESCE(hit_flag,0)=1 THEN COALESCE(paid,0) ELSE 0 END) / NULLIF(SUM(COALESCE(edited_lines,0)),0)
  display_name: Original Paid per Edit
  comment: Hit paid divided by edited lines, matching Tableau Orig Paid per Edit intent.
- name: GPV per Edit
  expr: SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(edited_lines,0)),0)
  display_name: GPV per Edit
  comment: GPV per edited line.
- name: NPV per Edit
  expr: SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(edited_lines,0)),0)
  display_name: NPV per Edit
  comment: NPV per edited line.
- name: Customer Count
  expr: COUNT(DISTINCT customer)
  display_name: Customer Count
  comment: Distinct customers in the current slice.
  synonyms:
  - customers
  - accounts
- name: Payer Count
  expr: COUNT(DISTINCT payer_short)
  display_name: Payer Count
  comment: Distinct payers in the current slice.
  synonyms:
  - payers
  - plans
- name: Super Payer Count
  expr: COUNT(DISTINCT super_payer_short)
  display_name: Super Payer Count
  comment: Distinct super payers in the current slice.
- name: DP Count
  expr: COUNT(DISTINCT dp_key)
  display_name: DP Count
  comment: Distinct decision points in the current slice.
  synonyms:
  - number of DPs
  - rules count
- name: MR Count
  expr: COUNT(DISTINCT mid_rule_key)
  display_name: MR Count
  comment: Distinct mid-rules in the current slice.
  synonyms:
  - number of MRs
  - mid rule count
- name: Medical Policy Count
  expr: COUNT(DISTINCT medical_policy)
  display_name: Medical Policy Count
  comment: Distinct medical policies in the current slice.
- name: Topic Count
  expr: COUNT(DISTINCT topic)
  display_name: Topic Count
  comment: Distinct topics in the current slice.
- name: Sub Rule Count
  expr: COUNT(DISTINCT sub_rule_key)
  display_name: Sub Rule Count
  comment: Distinct sub-rules in the current slice.
- name: Customer Adoption Rate
  expr: 100.0 * COUNT(DISTINCT customer) / NULLIF(ANY_VALUE(total_customers_universe),0)
  display_name: Customer Adoption Rate
  comment: Share of all benchmark customers represented in the current slice.
  synonyms:
  - customer adoption
  - adoption percent
- name: Payer Adoption Rate
  expr: 100.0 * COUNT(DISTINCT payer_short) / NULLIF(ANY_VALUE(total_payers_universe),0)
  display_name: Payer Adoption Rate
  comment: Share of all benchmark payers represented in the current slice.
  synonyms:
  - payer adoption
- name: Super Payer Adoption Rate
  expr: 100.0 * COUNT(DISTINCT super_payer_short) / NULLIF(ANY_VALUE(total_super_payers_universe),0)
  display_name: Super Payer Adoption Rate
  comment: Share of all benchmark super payers represented in the current slice.
  synonyms:
  - super payer adoption
- name: DP Adoption Rate
  expr: 100.0 * COUNT(DISTINCT dp_key) / NULLIF(ANY_VALUE(total_dps_universe),0)
  display_name: DP Adoption Rate
  comment: Share of all benchmark DPs represented in the current slice.
  synonyms:
  - DP adoption
- name: Hit Line Count
  expr: SUM(COALESCE(hit_flag,0))
  display_name: Hit Line Count
  comment: Sum of hit_flag.
- name: CV Flag Count
  expr: SUM(COALESCE(cv_flag,0))
  display_name: CV Flag Count
  comment: Sum of cv_flag.
- name: CV OP Active Client Flag Count
  expr: SUM(COALESCE(cv_op_flag_active_client,0))
  display_name: CV OP Active Client Flag Count
  comment: Sum of CV outpatient active client flag.
- name: CV Prof Active Client Flag Count
  expr: SUM(COALESCE(cv_prof_flag_active_client,0))
  display_name: CV Prof Active Client Flag Count
  comment: Sum of CV professional active client flag.
- name: Total CV Paid
  expr: SUM(COALESCE(cv_paid,0))
  display_name: Total CV Paid
  comment: CV paid dollars from source.
- name: Total Paid PPM
  expr: SUM(CASE WHEN tableau_solution_family = 'PPM' THEN COALESCE(paid,0) ELSE 0 END)
  display_name: Total Paid PPM
  comment: Tableau-style paid for PPM solution selection.
- name: Total Paid CV
  expr: SUM(COALESCE(cv_paid,0))
  display_name: Total Paid CV
  comment: Tableau-style paid for CV solution selection, using cv_paid.
- name: Total Paid FWA
  expr: SUM(CASE WHEN tableau_solution_family = 'FWA' THEN COALESCE(paid,0) ELSE 0 END)
  display_name: Total Paid FWA
  comment: Tableau-style paid for FWA solution selection.
- name: Total Paid CPR
  expr: SUM(CASE WHEN tableau_solution_family = 'CPR' THEN COALESCE(paid,0) ELSE 0 END)
  display_name: Total Paid CPR
  comment: Tableau-style paid for CPR solution selection.
- name: Total GPV PPM
  expr: SUM(CASE WHEN tableau_solution_family = 'PPM' THEN COALESCE(gpv,0) ELSE 0 END)
  display_name: Total GPV PPM
  comment: GPV filtered to PPM.
- name: Total GPV CV
  expr: SUM(CASE WHEN tableau_solution_family = 'CV' THEN COALESCE(gpv,0) ELSE 0 END)
  display_name: Total GPV CV
  comment: GPV filtered to CV.
- name: Total GPV FWA
  expr: SUM(CASE WHEN tableau_solution_family = 'FWA' THEN COALESCE(gpv,0) ELSE 0 END)
  display_name: Total GPV FWA
  comment: GPV filtered to FWA.
- name: Total GPV CPR
  expr: SUM(CASE WHEN tableau_solution_family = 'CPR' THEN COALESCE(gpv,0) ELSE 0 END)
  display_name: Total GPV CPR
  comment: GPV filtered to CPR.
- name: Total NPV PPM
  expr: SUM(CASE WHEN tableau_solution_family = 'PPM' THEN COALESCE(npv,0) ELSE 0 END)
  display_name: Total NPV PPM
  comment: NPV filtered to PPM.
- name: Total NPV CV
  expr: SUM(CASE WHEN tableau_solution_family = 'CV' THEN COALESCE(npv,0) ELSE 0 END)
  display_name: Total NPV CV
  comment: NPV filtered to CV.
- name: Total NPV FWA
  expr: SUM(CASE WHEN tableau_solution_family = 'FWA' THEN COALESCE(npv,0) ELSE 0 END)
  display_name: Total NPV FWA
  comment: NPV filtered to FWA.
- name: Total NPV CPR
  expr: SUM(CASE WHEN tableau_solution_family = 'CPR' THEN COALESCE(npv,0) ELSE 0 END)
  display_name: Total NPV CPR
  comment: NPV filtered to CPR.
- name: Active Rule NPV
  expr: SUM(CASE WHEN rule_status_label = 'Active' THEN COALESCE(npv,0) ELSE 0 END)
  display_name: Active Rule NPV
  comment: NPV from active rules only.
- name: Disabled Rule Unapplied
  expr: SUM(CASE WHEN COALESCE(disabled,'N') = 'Y' THEN COALESCE(unapplied,0) ELSE 0 END)
  display_name: Disabled Rule Unapplied
  comment: Unapplied value from disabled rules.
- name: Deactivated Rule Unapplied
  expr: SUM(CASE WHEN COALESCE(deactivated,'N') = 'Y' THEN COALESCE(unapplied,0) ELSE 0 END)
  display_name: Deactivated Rule Unapplied
  comment: Unapplied value from deactivated rules.
- name: Net Retention Rate
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(gpv,0)),0)
  display_name: Net Retention Rate
  comment: NPV divided by GPV.
- name: Hit Rate
  expr: 100.0 * SUM(COALESCE(hit_flag,0)) / NULLIF(COUNT(1),0)
  display_name: Hit Rate
  comment: Hit rows divided by row count.
- name: GPV Concentration
  expr: 100.0 * SUM(COALESCE(gpv,0)) / NULLIF(ANY_VALUE(total_gpv_universe),0)
  display_name: GPV Concentration
  comment: Share of total benchmark GPV represented by current slice.
  synonyms:
  - GPV share
  - GPV contribution
- name: NPV Concentration
  expr: 100.0 * SUM(COALESCE(npv,0)) / NULLIF(ANY_VALUE(total_npv_universe),0)
  display_name: NPV Concentration
  comment: Share of total benchmark NPV represented by current slice.
  synonyms:
  - NPV share
  - NPV contribution
- name: Paid Concentration Raw
  expr: 100.0 * SUM(COALESCE(paid,0)) / NULLIF(ANY_VALUE(total_paid_raw_universe),0)
  display_name: Paid Concentration Raw
  comment: Share of raw paid represented by current slice.
- name: Paid Concentration Dedup
  expr: 100.0 * SUM(COALESCE(paid_once_customer_payer_state,0)) / NULLIF(ANY_VALUE(total_paid_dedup_universe),0)
  display_name: Paid Concentration Dedup
  comment: Share of de-duplicated paid represented by current slice.
$$


## Dashboard Dimensions Catalog

This creates a temporary catalog table so reviewers can inspect every semantic dimension, its SQL expression, and its business definition.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW ppm_genie_dimension_catalog AS
SELECT * FROM VALUES
  ('Overall', '''Overall''', 'Constant dimension used to reproduce Tableau Overall level.'),
  ('Customer', 'customer', 'Customer/account from the PPM benchmark library.'),
  ('Customer Segment', 'customer_segment', 'Customer segment used in the dashboard Level 1/Level 2 selector.'),
  ('Line of Business', 'insurance_desc', 'LOB / insurance line, matching Tableau LOB.'),
  ('Product Type', 'product_type', 'Product type slicer.'),
  ('Claim Type', 'claim_type_desc_oi', 'Operational claim type used by the benchmark dashboard.'),
  ('Claim Type Base', 'claim_type_desc', 'Base claim type description.'),
  ('Final State', 'final_state', 'Final payer/customer state, used by the Tableau State parameter.'),
  ('Discrete State', 'discrete_state', 'Discrete state field from the source.'),
  ('Payer', 'payer_short', 'Payer short name.'),
  ('Payer Full Name', 'payer_name', 'Full payer name.'),
  ('Super Payer', 'super_payer_short', 'Super payer short name.'),
  ('Super Payer Full Name', 'super_payer_name', 'Full super payer name.'),
  ('Blues Indicator', 'blues_indicator_label', 'Blues vs Non-Blues label derived from blues_indicator.'),
  ('Blues Raw Flag', 'blues_indicator', 'Raw Y/N Blues flag.'),
  ('Blues Home Host', 'blues_home_host', 'Blues home/host indicator from source.'),
  ('IDN Indicator', 'idn_indicator_label', 'IDN vs Non-IDN label derived from idn_yn.'),
  ('Status', 'status', 'Payer lifecycle status, for current/retired filtering.'),
  ('Current Payer Flag', 'current_payer_flag', 'Y when payer status is Current; retained to mimic active payer dashboard filter.'),
  ('Rule Status', 'rule_status_label', 'Active, Disabled, or Deactivated rule status.'),
  ('Disabled', 'disabled', 'Raw disabled flag.'),
  ('Deactivated', 'deactivated', 'Raw deactivated flag.'),
  ('Rule Type', 'rule_type_label', 'Library Rule vs Custom Rule, matching Tableau rule type filter.'),
  ('Solution Family', 'tableau_solution_family', 'Tableau-compatible solution family derived from cv_rule_10 and solution.'),
  ('Source Solution', 'solution', 'Raw source solution value.'),
  ('CV Rule 10', 'cv_rule_10', 'CV rule 10 flag used by Tableau solution logic.'),
  ('Core Enhanced', 'core_enhanced_label', 'Core/Enhanced key/classification from Tableau source; use values available in prim_core_enhanced_key.'),
  ('Policy Collection Bundle', 'pcbi_master_list', 'Policy collection/bundle list used by Tableau Policy Collection/Bundle parameter.'),
  ('Policy Bundle Indicator', 'policy_bundle_indicator', 'Policy bundle indicator.'),
  ('Policy Package Indicator', 'policy_package_indicator', 'Policy package indicator.'),
  ('Policy Package Individual Indicator', 'policy_package_individual_indicator', 'Policy package individual indicator.'),
  ('Decision Point', 'dp_key', 'Decision Point key, the main DP dashboard grain.'),
  ('Decision Point Description', 'dp_desc', 'Decision Point description.'),
  ('DP Age Bucket', 'dp_age_bucket', 'Tableau DP Age in years bucket: Custom DP, Less than 1 Year, or whole years.'),
  ('DP Age Years', 'dp_age_years', 'Numeric DP age in years based on dp_start_date.'),
  ('DP Start Date', 'DATE(dp_start_date)', 'Decision Point start date.'),
  ('Medical Policy', 'medical_policy_label', 'Medical policy with Tableau null handling: null becomes ZZ.Null.'),
  ('Latest Medical Policy', 'latest_medical1_policy', 'Latest medical policy field.'),
  ('Topic', 'topic_label', 'Topic with Tableau null handling: null becomes ZZ.Null.'),
  ('Mid Rule Key', 'mid_rule_key_label', 'Mid Rule key with Tableau null handling.'),
  ('Mid Rule Version', 'mid_rule_version', 'Mid Rule version.'),
  ('Mid Rule Dot Version', 'mid_rule_dot_version', 'Mid Rule dot version.'),
  ('Sub Rule Key', 'sub_rule_key', 'Sub-rule key.'),
  ('Sub Rule Description', 'sub_rule_desc', 'Sub-rule description.'),
  ('DP for Sub Rule', 'dp_key_4_sub_rule_key', 'DP key associated to sub-rule key.'),
  ('DP Description for Sub Rule', 'dp_desc_4_sub_rule_key', 'DP description associated to sub-rule key.'),
  ('Topic Key for Sub Rule', 'topic_key_4_sub_rule_key', 'Topic key associated to sub-rule key.'),
  ('Topic for Sub Rule', 'topic_title_4_sub_rule_key', 'Topic title associated to sub-rule key.'),
  ('Medical Policy Key for Sub Rule', 'med_pol_key_4_sub_rule_key', 'Medical policy key associated to sub-rule key.'),
  ('Medical Policy for Sub Rule', 'med_pol_title_4_sub_rule_key', 'Medical policy title associated to sub-rule key.'),
  ('Policy Type Key for Sub Rule', 'pol_type_key_4_sub_rule_key', 'Policy type key associated to sub-rule key.'),
  ('Policy Type for Sub Rule', 'pol_type_desc_4_sub_rule_key', 'Policy type description associated to sub-rule key.'),
  ('Claims Platform', 'claims_platform_label', 'Claims platform with Tableau Unknown handling.'),
  ('Cotiviti Edit Position', 'cotiviti_edit_position_label', 'Cotiviti edit position normalized to 1-5 or Unknown.'),
  ('Primary 3rd Party Editor', 'primary_editor_label', 'Primary third-party editor with Tableau Unknown handling.'),
  ('Secondary 3rd Party Editor', 'secondary_3rd_party_editor', 'Secondary third-party editor.'),
  ('Has Competitor', 'CASE WHEN primary_editor_label = ''Unknown'' THEN ''No Competitor'' ELSE ''Has Competitor'' END', 'Whether a primary third-party editor is present.'),
  ('SVP', 'svp', 'SVP owner dimension.'),
  ('CEL', 'cel', 'Client engagement lead dimension.'),
  ('CSD', 'csd', 'CSD owner dimension.'),
  ('CSM', 'csm', 'CSM owner dimension.'),
  ('CPM', 'cpm', 'CPM owner dimension.'),
  ('BA', 'ba', 'BA owner dimension.'),
  ('Client Medical Director', 'client_medical_director', 'Client medical director.'),
  ('Min Invoice Date', 'DATE(min_invoice_date)', 'Minimum invoice date.'),
  ('Max Invoice Date', 'DATE(max_invoice_date)', 'Maximum invoice date.'),
  ('Payer Age Years', 'payer_age_years', 'Payer age in years from min_invoice_date.'),
  ('Max Invoice Year', 'max_invoice_year', 'Maximum invoice year.'),
  ('Latest Ref Date', 'latest_ref_date', 'Latest data refresh/reference date.'),
  ('Insurance Key', 'insurance_key', 'Insurance key.'),
  ('Customer Code', 'customer_code', 'Customer code.'),
  ('Payer Key', 'payer_key', 'Payer key.'),
  ('Payer Short Code', 'payer_short_code', 'Payer short code.'),
  ('Super Payer Key', 'super_payer_key', 'Super payer key.'),
  ('Super Payer Code', 'super_payer_code', 'Super payer code.'),
  ('CV Code', 'cv_code', 'CV code.'),
  ('CV Client', 'cv_client', 'CV client flag.'),
  ('NDC Data', 'ndc_data', 'NDC data flag.'),
  ('Client Name', 'client_name', 'Client name from source.'),
  ('PCI Master List', 'pci_master_list', 'PCI master list.'),
  ('PBI Master List', 'pbi_master_list', 'PBI master list.'),
  ('PCBI Master List', 'pcbi_master_list', 'Combined PCBI master list.')
AS t(dimension_name, sql_expression, business_definition);

SELECT * FROM ppm_genie_dimension_catalog
ORDER BY dimension_name;


## Complete Metric Definition Catalog

This is exhaustive. It includes every metric in the metric view: core KPIs, LOD denominators, level-specific percentages, counts, adoption metrics, solution-specific metrics, and concentration metrics.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW ppm_genie_metric_definition_catalog AS
SELECT * FROM VALUES
  ('All / Overall', 'Total Paid Raw', 'SUM(COALESCE(paid,0))', 'Use only when intentionally matching raw Tableau/raw source paid behavior; avoid for DP/MR/policy/topic fan-out.', 'Raw sum of paid. Use mainly for Level 1 dashboard cuts where paid is not fanned out.'),
  ('All / Overall', 'Total Paid Dedup Customer Payer State', 'SUM(COALESCE(paid_once_customer_payer_state,0))', 'General benchmark metric.', 'Paid counted once per customer x payer x final state. This is the safest overall paid denominator when DP/MR/policy fan-out is present.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid Customer Segment Denominator', 'SUM(COALESCE(paid_once_customer_segment,0))', 'Use when grouping or filtering by Customer Segment.', 'Paid denominator counted once per customer x payer x final state x customer segment. Use for Overall Benchmark customer segment cuts.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid LOB Denominator', 'SUM(COALESCE(paid_once_lob,0))', 'Use when grouping or filtering by Line of Business / LOB.', 'Paid denominator counted once per customer x payer x final state x line of business. Use for Overall Benchmark LOB cuts.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid Product Type Denominator', 'SUM(COALESCE(paid_once_product_type,0))', 'Use when grouping or filtering by Product Type.', 'Paid denominator counted once per customer x payer x final state x product type. Use for Overall Benchmark product type cuts.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid Claim Type Denominator', 'SUM(COALESCE(paid_once_claim_type,0))', 'Use when grouping or filtering by Claim Type.', 'Paid denominator counted once per customer x payer x final state x claim type. Use for Overall Benchmark claim type cuts.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid Blues Denominator', 'SUM(COALESCE(paid_once_blues,0))', 'Use when grouping or filtering by Blues Indicator.', 'Paid denominator counted once per customer x payer x final state x Blues indicator. Use for Overall Benchmark Blues vs Non-Blues cuts.'),
  ('DP Benchmark', 'Total Paid DP Denominator', 'SUM(COALESCE(paid_once_dp,0))', 'Use for DP Benchmark and any query including Decision Point.', 'Paid denominator counted once per customer x payer x final state x DP. Use for DP-level percentage questions.'),
  ('MR Benchmark', 'Total Paid MR Denominator', 'SUM(COALESCE(paid_once_mid_rule,0))', 'Use for MR Benchmark and any query including Mid Rule Key.', 'Paid denominator counted once per customer x payer x final state x MR. Use for MR-level percentage questions.'),
  ('Medical Policy Benchmark', 'Total Paid Medical Policy Denominator', 'SUM(COALESCE(paid_once_medical_policy,0))', 'Use for Policy-Topic Benchmark when Medical Policy is selected.', 'Paid denominator counted once per customer x payer x final state x medical policy. Use for policy-level percentage questions.'),
  ('Topic Benchmark', 'Total Paid Topic Denominator', 'SUM(COALESCE(paid_once_topic,0))', 'Use for Policy-Topic Benchmark when Topic is selected.', 'Paid denominator counted once per customer x payer x final state x topic. Use for topic-level percentage questions.'),
  ('Sub Rule Drilldown', 'Total Paid Sub Rule Denominator', 'SUM(COALESCE(paid_once_sub_rule,0))', 'Use for sub-rule detail drilldowns.', 'Paid denominator counted once per customer x payer x final state x sub-rule. Use for sub-rule-level percentage questions.'),
  ('All / Overall', 'Total GPV', 'SUM(COALESCE(gpv,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Gross payment value / gross savings.'),
  ('All / Overall', 'Total Adjustments', 'SUM(COALESCE(adjustments,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Adjustment dollars applied against GPV.'),
  ('All / Overall', 'Total NPV', 'SUM(COALESCE(npv,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Net payment value / net savings.'),
  ('All / Overall', 'Total EPV', 'SUM(COALESCE(epv,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Expected payment value from source.'),
  ('All / Overall', 'Total Unapplied', 'SUM(COALESCE(unapplied,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Unapplied savings opportunity.'),
  ('All / Overall', 'GPV Percent Raw', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid,0)),0)', 'Use only when intentionally matching raw Tableau/raw source paid behavior; avoid for DP/MR/policy/topic fan-out.', 'Raw GPV percent. Correct only when paid is not duplicated by the selected dimensions.'),
  ('All / Overall', 'NPV Percent Raw', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid,0)),0)', 'Use only when intentionally matching raw Tableau/raw source paid behavior; avoid for DP/MR/policy/topic fan-out.', 'Raw NPV percent. Correct only when paid is not duplicated by the selected dimensions.'),
  ('All / Overall', 'GPV Percent Overall LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_payer_state,0)),0)', 'Use for scorecard, state, payer, customer, and overall cuts when no DP/MR/policy/topic fan-out is requested.', 'LOD-safe GPV percent using customer-payer-state paid denominator.'),
  ('All / Overall', 'NPV Percent Overall LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_payer_state,0)),0)', 'Use for scorecard, state, payer, customer, and overall cuts when no DP/MR/policy/topic fan-out is requested.', 'LOD-safe NPV percent using customer-payer-state paid denominator.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by Customer Segment LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_segment,0)),0)', 'Use when grouping or filtering by Customer Segment.', 'Use when grouping/filtering by Customer Segment in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by Customer Segment LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_segment,0)),0)', 'Use when grouping or filtering by Customer Segment.', 'Use when grouping/filtering by Customer Segment in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by LOB LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_lob,0)),0)', 'Use when grouping or filtering by Line of Business / LOB.', 'Use when grouping/filtering by Line of Business in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by LOB LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_lob,0)),0)', 'Use when grouping or filtering by Line of Business / LOB.', 'Use when grouping/filtering by Line of Business in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by Product Type LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_product_type,0)),0)', 'Use when grouping or filtering by Product Type.', 'Use when grouping/filtering by Product Type in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by Product Type LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_product_type,0)),0)', 'Use when grouping or filtering by Product Type.', 'Use when grouping/filtering by Product Type in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by Claim Type LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_claim_type,0)),0)', 'Use when grouping or filtering by Claim Type.', 'Use when grouping/filtering by Claim Type in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by Claim Type LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_claim_type,0)),0)', 'Use when grouping or filtering by Claim Type.', 'Use when grouping/filtering by Claim Type in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by Blues LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_blues,0)),0)', 'Use when grouping or filtering by Blues Indicator.', 'Use when grouping/filtering by Blues Indicator in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by Blues LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_blues,0)),0)', 'Use when grouping or filtering by Blues Indicator.', 'Use when grouping/filtering by Blues Indicator in the Overall Benchmark.'),
  ('DP Benchmark', 'GPV Percent by DP LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_dp,0)),0)', 'Use for DP Benchmark and any query including Decision Point.', 'Use when grouping/filtering by Decision Point.'),
  ('DP Benchmark', 'NPV Percent by DP LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_dp,0)),0)', 'Use for DP Benchmark and any query including Decision Point.', 'Use when grouping/filtering by Decision Point.'),
  ('MR Benchmark', 'GPV Percent by MR LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_mid_rule,0)),0)', 'Use for MR Benchmark and any query including Mid Rule Key.', 'Use when grouping/filtering by Mid Rule.'),
  ('MR Benchmark', 'NPV Percent by MR LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_mid_rule,0)),0)', 'Use for MR Benchmark and any query including Mid Rule Key.', 'Use when grouping/filtering by Mid Rule.'),
  ('Medical Policy Benchmark', 'GPV Percent by Medical Policy LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_medical_policy,0)),0)', 'Use for Policy-Topic Benchmark when Medical Policy is selected.', 'Use when grouping/filtering by Medical Policy.'),
  ('Medical Policy Benchmark', 'NPV Percent by Medical Policy LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_medical_policy,0)),0)', 'Use for Policy-Topic Benchmark when Medical Policy is selected.', 'Use when grouping/filtering by Medical Policy.'),
  ('Topic Benchmark', 'GPV Percent by Topic LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_topic,0)),0)', 'Use for Policy-Topic Benchmark when Topic is selected.', 'Use when grouping/filtering by Topic.'),
  ('Topic Benchmark', 'NPV Percent by Topic LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_topic,0)),0)', 'Use for Policy-Topic Benchmark when Topic is selected.', 'Use when grouping/filtering by Topic.'),
  ('Sub Rule Drilldown', 'GPV Percent by Sub Rule LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_sub_rule,0)),0)', 'Use for sub-rule detail drilldowns.', 'Use when grouping/filtering by Sub Rule.'),
  ('Sub Rule Drilldown', 'NPV Percent by Sub Rule LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_sub_rule,0)),0)', 'Use for sub-rule detail drilldowns.', 'Use when grouping/filtering by Sub Rule.'),
  ('All / Overall', 'APV Percent', '100.0 * SUM(COALESCE(adjustments,0)) / NULLIF(SUM(COALESCE(gpv,0)),0)', 'General benchmark metric.', 'Adjustment percent: adjustments divided by GPV, matching Tableau APV%.'),
  ('All / Overall', 'Unapplied Percent', '100.0 * SUM(COALESCE(unapplied,0)) / NULLIF(SUM(COALESCE(unapplied,0)) + SUM(COALESCE(gpv,0)),0)', 'General benchmark metric.', 'Unapplied divided by unapplied plus GPV, matching Tableau Unapplied%.'),
  ('All / Overall', 'Total Lines', 'SUM(COALESCE(total_lines,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Total claim/service lines.'),
  ('All / Overall', 'Edited Lines', 'SUM(COALESCE(edited_lines,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Edited lines.'),
  ('All / Overall', 'Adjusted Lines', 'SUM(COALESCE(adjusted_lines,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Adjusted lines.'),
  ('All / Overall', 'Adjusted Lines Percent', '100.0 * SUM(COALESCE(adjusted_lines,0)) / NULLIF(SUM(COALESCE(edited_lines,0)),0)', 'General benchmark metric.', 'Adjusted lines divided by edited lines.'),
  ('All / Overall', 'Edits per 1000', '1000.0 * SUM(COALESCE(edited_lines,0)) / NULLIF(SUM(COALESCE(total_lines,0)),0)', 'General benchmark metric.', 'Edited lines per 1,000 total lines.'),
  ('All / Overall', 'Original Paid per TBA Line', 'SUM(COALESCE(paid,0)) / NULLIF(SUM(COALESCE(total_lines,0)),0)', 'General benchmark metric.', 'Paid dollars per total/TBA line.'),
  ('All / Overall', 'Original Paid per Edit', 'SUM(CASE WHEN COALESCE(hit_flag,0)=1 THEN COALESCE(paid,0) ELSE 0 END) / NULLIF(SUM(COALESCE(edited_lines,0)),0)', 'General benchmark metric.', 'Hit paid divided by edited lines, matching Tableau Orig Paid per Edit intent.'),
  ('All / Overall', 'GPV per Edit', 'SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(edited_lines,0)),0)', 'General benchmark metric.', 'GPV per edited line.'),
  ('All / Overall', 'NPV per Edit', 'SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(edited_lines,0)),0)', 'General benchmark metric.', 'NPV per edited line.'),
  ('All / Overall', 'Customer Count', 'COUNT(DISTINCT customer)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct customers in the current slice.'),
  ('All / Overall', 'Payer Count', 'COUNT(DISTINCT payer_short)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct payers in the current slice.'),
  ('All / Overall', 'Super Payer Count', 'COUNT(DISTINCT super_payer_short)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct super payers in the current slice.'),
  ('DP Benchmark', 'DP Count', 'COUNT(DISTINCT dp_key)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct decision points in the current slice.'),
  ('MR Benchmark', 'MR Count', 'COUNT(DISTINCT mid_rule_key)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct mid-rules in the current slice.'),
  ('Medical Policy Benchmark', 'Medical Policy Count', 'COUNT(DISTINCT medical_policy)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct medical policies in the current slice.'),
  ('Topic Benchmark', 'Topic Count', 'COUNT(DISTINCT topic)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct topics in the current slice.'),
  ('Sub Rule Drilldown', 'Sub Rule Count', 'COUNT(DISTINCT sub_rule_key)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct sub-rules in the current slice.'),
  ('All / Overall', 'Customer Adoption Rate', '100.0 * COUNT(DISTINCT customer) / NULLIF(ANY_VALUE(total_customers_universe),0)', 'General benchmark metric.', 'Share of all benchmark customers represented in the current slice.'),
  ('All / Overall', 'Payer Adoption Rate', '100.0 * COUNT(DISTINCT payer_short) / NULLIF(ANY_VALUE(total_payers_universe),0)', 'General benchmark metric.', 'Share of all benchmark payers represented in the current slice.'),
  ('All / Overall', 'Super Payer Adoption Rate', '100.0 * COUNT(DISTINCT super_payer_short) / NULLIF(ANY_VALUE(total_super_payers_universe),0)', 'General benchmark metric.', 'Share of all benchmark super payers represented in the current slice.'),
  ('DP Benchmark', 'DP Adoption Rate', '100.0 * COUNT(DISTINCT dp_key) / NULLIF(ANY_VALUE(total_dps_universe),0)', 'General benchmark metric.', 'Share of all benchmark DPs represented in the current slice.'),
  ('All / Overall', 'Hit Line Count', 'SUM(COALESCE(hit_flag,0))', 'General benchmark metric.', 'Sum of hit_flag.'),
  ('Solution Slice', 'CV Flag Count', 'SUM(COALESCE(cv_flag,0))', 'Use when user filters or compares solution family.', 'Sum of cv_flag.'),
  ('Solution Slice', 'CV OP Active Client Flag Count', 'SUM(COALESCE(cv_op_flag_active_client,0))', 'Use when user filters or compares solution family.', 'Sum of CV outpatient active client flag.'),
  ('Solution Slice', 'CV Prof Active Client Flag Count', 'SUM(COALESCE(cv_prof_flag_active_client,0))', 'Use when user filters or compares solution family.', 'Sum of CV professional active client flag.'),
  ('Solution Slice', 'Total CV Paid', 'SUM(COALESCE(cv_paid,0))', 'Use when user filters or compares solution family.', 'CV paid dollars from source.'),
  ('Solution Slice', 'Total Paid PPM', 'SUM(CASE WHEN tableau_solution_family = ''PPM'' THEN COALESCE(paid,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'Tableau-style paid for PPM solution selection.'),
  ('Solution Slice', 'Total Paid CV', 'SUM(COALESCE(cv_paid,0))', 'Use when user filters or compares solution family.', 'Tableau-style paid for CV solution selection, using cv_paid.'),
  ('Solution Slice', 'Total Paid FWA', 'SUM(CASE WHEN tableau_solution_family = ''FWA'' THEN COALESCE(paid,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'Tableau-style paid for FWA solution selection.'),
  ('Solution Slice', 'Total Paid CPR', 'SUM(CASE WHEN tableau_solution_family = ''CPR'' THEN COALESCE(paid,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'Tableau-style paid for CPR solution selection.'),
  ('Solution Slice', 'Total GPV PPM', 'SUM(CASE WHEN tableau_solution_family = ''PPM'' THEN COALESCE(gpv,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'GPV filtered to PPM.'),
  ('Solution Slice', 'Total GPV CV', 'SUM(CASE WHEN tableau_solution_family = ''CV'' THEN COALESCE(gpv,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'GPV filtered to CV.'),
  ('Solution Slice', 'Total GPV FWA', 'SUM(CASE WHEN tableau_solution_family = ''FWA'' THEN COALESCE(gpv,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'GPV filtered to FWA.'),
  ('Solution Slice', 'Total GPV CPR', 'SUM(CASE WHEN tableau_solution_family = ''CPR'' THEN COALESCE(gpv,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'GPV filtered to CPR.'),
  ('Solution Slice', 'Total NPV PPM', 'SUM(CASE WHEN tableau_solution_family = ''PPM'' THEN COALESCE(npv,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'NPV filtered to PPM.'),
  ('Solution Slice', 'Total NPV CV', 'SUM(CASE WHEN tableau_solution_family = ''CV'' THEN COALESCE(npv,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'NPV filtered to CV.'),
  ('Solution Slice', 'Total NPV FWA', 'SUM(CASE WHEN tableau_solution_family = ''FWA'' THEN COALESCE(npv,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'NPV filtered to FWA.'),
  ('Solution Slice', 'Total NPV CPR', 'SUM(CASE WHEN tableau_solution_family = ''CPR'' THEN COALESCE(npv,0) ELSE 0 END)', 'Use when user filters or compares solution family.', 'NPV filtered to CPR.'),
  ('All / Overall', 'Active Rule NPV', 'SUM(CASE WHEN rule_status_label = ''Active'' THEN COALESCE(npv,0) ELSE 0 END)', 'General benchmark metric.', 'NPV from active rules only.'),
  ('All / Overall', 'Disabled Rule Unapplied', 'SUM(CASE WHEN COALESCE(disabled,''N'') = ''Y'' THEN COALESCE(unapplied,0) ELSE 0 END)', 'General benchmark metric.', 'Unapplied value from disabled rules.'),
  ('All / Overall', 'Deactivated Rule Unapplied', 'SUM(CASE WHEN COALESCE(deactivated,''N'') = ''Y'' THEN COALESCE(unapplied,0) ELSE 0 END)', 'General benchmark metric.', 'Unapplied value from deactivated rules.'),
  ('All / Overall', 'Net Retention Rate', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(gpv,0)),0)', 'General benchmark metric.', 'NPV divided by GPV.'),
  ('All / Overall', 'Hit Rate', '100.0 * SUM(COALESCE(hit_flag,0)) / NULLIF(COUNT(1),0)', 'General benchmark metric.', 'Hit rows divided by row count.'),
  ('All / Overall', 'GPV Concentration', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(ANY_VALUE(total_gpv_universe),0)', 'General benchmark metric.', 'Share of total benchmark GPV represented by current slice.'),
  ('All / Overall', 'NPV Concentration', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(ANY_VALUE(total_npv_universe),0)', 'General benchmark metric.', 'Share of total benchmark NPV represented by current slice.'),
  ('All / Overall', 'Paid Concentration Raw', '100.0 * SUM(COALESCE(paid,0)) / NULLIF(ANY_VALUE(total_paid_raw_universe),0)', 'Use only when intentionally matching raw Tableau/raw source paid behavior; avoid for DP/MR/policy/topic fan-out.', 'Share of raw paid represented by current slice.'),
  ('All / Overall', 'Paid Concentration Dedup', '100.0 * SUM(COALESCE(paid_once_customer_payer_state,0)) / NULLIF(ANY_VALUE(total_paid_dedup_universe),0)', 'General benchmark metric.', 'Share of de-duplicated paid represented by current slice.')
AS t(level_group, metric_name, sql_definition, genie_usage, business_definition);

SELECT * FROM ppm_genie_metric_definition_catalog
ORDER BY level_group, metric_name;


## Core Dashboard Metrics Catalog

This narrower catalog contains the main dashboard-facing metric definitions that Genie should prefer for typical business questions. It includes the approximately 30 core KPI families plus the level-specific LOD-safe variants needed for Tableau parity.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW ppm_genie_core_dashboard_metrics AS
SELECT * FROM VALUES
  ('All / Overall', 'Total Paid Dedup Customer Payer State', 'SUM(COALESCE(paid_once_customer_payer_state,0))', 'General benchmark metric.', 'Paid counted once per customer x payer x final state. This is the safest overall paid denominator when DP/MR/policy fan-out is present.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid Customer Segment Denominator', 'SUM(COALESCE(paid_once_customer_segment,0))', 'Use when grouping or filtering by Customer Segment.', 'Paid denominator counted once per customer x payer x final state x customer segment. Use for Overall Benchmark customer segment cuts.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid LOB Denominator', 'SUM(COALESCE(paid_once_lob,0))', 'Use when grouping or filtering by Line of Business / LOB.', 'Paid denominator counted once per customer x payer x final state x line of business. Use for Overall Benchmark LOB cuts.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid Product Type Denominator', 'SUM(COALESCE(paid_once_product_type,0))', 'Use when grouping or filtering by Product Type.', 'Paid denominator counted once per customer x payer x final state x product type. Use for Overall Benchmark product type cuts.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid Claim Type Denominator', 'SUM(COALESCE(paid_once_claim_type,0))', 'Use when grouping or filtering by Claim Type.', 'Paid denominator counted once per customer x payer x final state x claim type. Use for Overall Benchmark claim type cuts.'),
  ('Overall Benchmark Dimension Cut', 'Total Paid Blues Denominator', 'SUM(COALESCE(paid_once_blues,0))', 'Use when grouping or filtering by Blues Indicator.', 'Paid denominator counted once per customer x payer x final state x Blues indicator. Use for Overall Benchmark Blues vs Non-Blues cuts.'),
  ('DP Benchmark', 'Total Paid DP Denominator', 'SUM(COALESCE(paid_once_dp,0))', 'Use for DP Benchmark and any query including Decision Point.', 'Paid denominator counted once per customer x payer x final state x DP. Use for DP-level percentage questions.'),
  ('MR Benchmark', 'Total Paid MR Denominator', 'SUM(COALESCE(paid_once_mid_rule,0))', 'Use for MR Benchmark and any query including Mid Rule Key.', 'Paid denominator counted once per customer x payer x final state x MR. Use for MR-level percentage questions.'),
  ('Medical Policy Benchmark', 'Total Paid Medical Policy Denominator', 'SUM(COALESCE(paid_once_medical_policy,0))', 'Use for Policy-Topic Benchmark when Medical Policy is selected.', 'Paid denominator counted once per customer x payer x final state x medical policy. Use for policy-level percentage questions.'),
  ('Topic Benchmark', 'Total Paid Topic Denominator', 'SUM(COALESCE(paid_once_topic,0))', 'Use for Policy-Topic Benchmark when Topic is selected.', 'Paid denominator counted once per customer x payer x final state x topic. Use for topic-level percentage questions.'),
  ('All / Overall', 'Total GPV', 'SUM(COALESCE(gpv,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Gross payment value / gross savings.'),
  ('All / Overall', 'Total Adjustments', 'SUM(COALESCE(adjustments,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Adjustment dollars applied against GPV.'),
  ('All / Overall', 'Total NPV', 'SUM(COALESCE(npv,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Net payment value / net savings.'),
  ('All / Overall', 'Total Unapplied', 'SUM(COALESCE(unapplied,0))', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Unapplied savings opportunity.'),
  ('All / Overall', 'GPV Percent Overall LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_payer_state,0)),0)', 'Use for scorecard, state, payer, customer, and overall cuts when no DP/MR/policy/topic fan-out is requested.', 'LOD-safe GPV percent using customer-payer-state paid denominator.'),
  ('All / Overall', 'NPV Percent Overall LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_payer_state,0)),0)', 'Use for scorecard, state, payer, customer, and overall cuts when no DP/MR/policy/topic fan-out is requested.', 'LOD-safe NPV percent using customer-payer-state paid denominator.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by Customer Segment LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_segment,0)),0)', 'Use when grouping or filtering by Customer Segment.', 'Use when grouping/filtering by Customer Segment in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by Customer Segment LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_customer_segment,0)),0)', 'Use when grouping or filtering by Customer Segment.', 'Use when grouping/filtering by Customer Segment in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by LOB LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_lob,0)),0)', 'Use when grouping or filtering by Line of Business / LOB.', 'Use when grouping/filtering by Line of Business in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by LOB LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_lob,0)),0)', 'Use when grouping or filtering by Line of Business / LOB.', 'Use when grouping/filtering by Line of Business in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by Product Type LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_product_type,0)),0)', 'Use when grouping or filtering by Product Type.', 'Use when grouping/filtering by Product Type in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by Product Type LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_product_type,0)),0)', 'Use when grouping or filtering by Product Type.', 'Use when grouping/filtering by Product Type in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by Claim Type LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_claim_type,0)),0)', 'Use when grouping or filtering by Claim Type.', 'Use when grouping/filtering by Claim Type in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by Claim Type LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_claim_type,0)),0)', 'Use when grouping or filtering by Claim Type.', 'Use when grouping/filtering by Claim Type in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'GPV Percent by Blues LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_blues,0)),0)', 'Use when grouping or filtering by Blues Indicator.', 'Use when grouping/filtering by Blues Indicator in the Overall Benchmark.'),
  ('Overall Benchmark Dimension Cut', 'NPV Percent by Blues LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_blues,0)),0)', 'Use when grouping or filtering by Blues Indicator.', 'Use when grouping/filtering by Blues Indicator in the Overall Benchmark.'),
  ('DP Benchmark', 'GPV Percent by DP LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_dp,0)),0)', 'Use for DP Benchmark and any query including Decision Point.', 'Use when grouping/filtering by Decision Point.'),
  ('DP Benchmark', 'NPV Percent by DP LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_dp,0)),0)', 'Use for DP Benchmark and any query including Decision Point.', 'Use when grouping/filtering by Decision Point.'),
  ('MR Benchmark', 'GPV Percent by MR LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_mid_rule,0)),0)', 'Use for MR Benchmark and any query including Mid Rule Key.', 'Use when grouping/filtering by Mid Rule.'),
  ('MR Benchmark', 'NPV Percent by MR LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_mid_rule,0)),0)', 'Use for MR Benchmark and any query including Mid Rule Key.', 'Use when grouping/filtering by Mid Rule.'),
  ('Medical Policy Benchmark', 'GPV Percent by Medical Policy LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_medical_policy,0)),0)', 'Use for Policy-Topic Benchmark when Medical Policy is selected.', 'Use when grouping/filtering by Medical Policy.'),
  ('Medical Policy Benchmark', 'NPV Percent by Medical Policy LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_medical_policy,0)),0)', 'Use for Policy-Topic Benchmark when Medical Policy is selected.', 'Use when grouping/filtering by Medical Policy.'),
  ('Topic Benchmark', 'GPV Percent by Topic LOD', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(SUM(COALESCE(paid_once_topic,0)),0)', 'Use for Policy-Topic Benchmark when Topic is selected.', 'Use when grouping/filtering by Topic.'),
  ('Topic Benchmark', 'NPV Percent by Topic LOD', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(SUM(COALESCE(paid_once_topic,0)),0)', 'Use for Policy-Topic Benchmark when Topic is selected.', 'Use when grouping/filtering by Topic.'),
  ('All / Overall', 'APV Percent', '100.0 * SUM(COALESCE(adjustments,0)) / NULLIF(SUM(COALESCE(gpv,0)),0)', 'General benchmark metric.', 'Adjustment percent: adjustments divided by GPV, matching Tableau APV%.'),
  ('All / Overall', 'Unapplied Percent', '100.0 * SUM(COALESCE(unapplied,0)) / NULLIF(SUM(COALESCE(unapplied,0)) + SUM(COALESCE(gpv,0)),0)', 'General benchmark metric.', 'Unapplied divided by unapplied plus GPV, matching Tableau Unapplied%.'),
  ('All / Overall', 'Edits per 1000', '1000.0 * SUM(COALESCE(edited_lines,0)) / NULLIF(SUM(COALESCE(total_lines,0)),0)', 'General benchmark metric.', 'Edited lines per 1,000 total lines.'),
  ('All / Overall', 'Adjusted Lines Percent', '100.0 * SUM(COALESCE(adjusted_lines,0)) / NULLIF(SUM(COALESCE(edited_lines,0)),0)', 'General benchmark metric.', 'Adjusted lines divided by edited lines.'),
  ('All / Overall', 'Original Paid per Edit', 'SUM(CASE WHEN COALESCE(hit_flag,0)=1 THEN COALESCE(paid,0) ELSE 0 END) / NULLIF(SUM(COALESCE(edited_lines,0)),0)', 'General benchmark metric.', 'Hit paid divided by edited lines, matching Tableau Orig Paid per Edit intent.'),
  ('All / Overall', 'Original Paid per TBA Line', 'SUM(COALESCE(paid,0)) / NULLIF(SUM(COALESCE(total_lines,0)),0)', 'General benchmark metric.', 'Paid dollars per total/TBA line.'),
  ('All / Overall', 'Customer Count', 'COUNT(DISTINCT customer)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct customers in the current slice.'),
  ('All / Overall', 'Payer Count', 'COUNT(DISTINCT payer_short)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct payers in the current slice.'),
  ('DP Benchmark', 'DP Count', 'COUNT(DISTINCT dp_key)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct decision points in the current slice.'),
  ('MR Benchmark', 'MR Count', 'COUNT(DISTINCT mid_rule_key)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct mid-rules in the current slice.'),
  ('Medical Policy Benchmark', 'Medical Policy Count', 'COUNT(DISTINCT medical_policy)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct medical policies in the current slice.'),
  ('Topic Benchmark', 'Topic Count', 'COUNT(DISTINCT topic)', 'Use as base KPI at any level; pair percentage metrics with level-specific LOD denominator.', 'Distinct topics in the current slice.'),
  ('All / Overall', 'Customer Adoption Rate', '100.0 * COUNT(DISTINCT customer) / NULLIF(ANY_VALUE(total_customers_universe),0)', 'General benchmark metric.', 'Share of all benchmark customers represented in the current slice.'),
  ('All / Overall', 'Payer Adoption Rate', '100.0 * COUNT(DISTINCT payer_short) / NULLIF(ANY_VALUE(total_payers_universe),0)', 'General benchmark metric.', 'Share of all benchmark payers represented in the current slice.'),
  ('All / Overall', 'GPV Concentration', '100.0 * SUM(COALESCE(gpv,0)) / NULLIF(ANY_VALUE(total_gpv_universe),0)', 'General benchmark metric.', 'Share of total benchmark GPV represented by current slice.'),
  ('All / Overall', 'NPV Concentration', '100.0 * SUM(COALESCE(npv,0)) / NULLIF(ANY_VALUE(total_npv_universe),0)', 'General benchmark metric.', 'Share of total benchmark NPV represented by current slice.')
AS t(level_group, metric_name, sql_definition, genie_usage, business_definition);

SELECT * FROM ppm_genie_core_dashboard_metrics
ORDER BY level_group, metric_name;


# LOD-Safe Metric Selection Guide

| User asks for / grouping includes | Use GPV metric | Use NPV metric | Denominator |
| --- | --- | --- | --- |
| Overall scorecard, Customer, Payer, State | `GPV Percent Overall LOD` | `NPV Percent Overall LOD` | `Total Paid Dedup Customer Payer State` |
| Customer Segment | `GPV Percent by Customer Segment LOD` | `NPV Percent by Customer Segment LOD` | `Total Paid Customer Segment Denominator` |
| LOB / Line of Business | `GPV Percent by LOB LOD` | `NPV Percent by LOB LOD` | `Total Paid LOB Denominator` |
| Product Type | `GPV Percent by Product Type LOD` | `NPV Percent by Product Type LOD` | `Total Paid Product Type Denominator` |
| Claim Type | `GPV Percent by Claim Type LOD` | `NPV Percent by Claim Type LOD` | `Total Paid Claim Type Denominator` |
| Blues Indicator | `GPV Percent by Blues LOD` | `NPV Percent by Blues LOD` | `Total Paid Blues Denominator` |
| DP Key / Decision Point | `GPV Percent by DP LOD` | `NPV Percent by DP LOD` | `Total Paid DP Denominator` |
| Mid Rule Key / MR | `GPV Percent by MR LOD` | `NPV Percent by MR LOD` | `Total Paid MR Denominator` |
| Medical Policy | `GPV Percent by Medical Policy LOD` | `NPV Percent by Medical Policy LOD` | `Total Paid Medical Policy Denominator` |
| Topic | `GPV Percent by Topic LOD` | `NPV Percent by Topic LOD` | `Total Paid Topic Denominator` |
| Sub Rule | `GPV Percent by Sub Rule LOD` | `NPV Percent by Sub Rule LOD` | `Total Paid Sub Rule Denominator` |

Genie instruction: when a user question includes one of these dimensions, choose the matching LOD-safe percentage metric instead of raw percent.


## Query Samples Across All Dashboard Levels

The following cells are runnable metric-view SQL examples covering:

- Overall scorecard
- Overall Benchmark dimensions: Customer Segment, LOB, Product Type, Claim Type, Blues, State
- Level 1 x Level 2 template
- DP Benchmark
- MR Benchmark
- Medical Policy Benchmark
- Topic Benchmark
- Policy/Topic drilldowns to DP and MR
- Solution and filter QA


```sql
-- PPM Benchmark Library - Metric Query Pack for Genie Validation
-- Metric view:
--   oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
--
-- These queries are organized by Tableau dashboard grain:
--   1. Overall Benchmark dimensions
--   2. DP Benchmark / Decision Point level
--   3. MR Benchmark / Mid Rule level
--   4. Policy-Topic Benchmark / Medical Policy and Topic level
--
-- Optional dashboard filters to add to any query:
--   WHERE `Solution Family` = 'PPM'                 -- or CV, FWA, CPR
--   WHERE `Final State` = 'CA'
--   WHERE `Rule Type` = 'Library Rule'              -- or Custom Rule
--   WHERE `Rule Status` = 'Active'                  -- or Disabled, Deactivated
--   WHERE `Policy Collection Bundle` ILIKE '%...%'
--   WHERE `DP Age Bucket` = 'Less than 1 Year'
--   WHERE `Core Enhanced` = 'Core'
```


In [ ]:
%sql
-- ================================================================
-- 0. OVERALL SCORECARD
-- Tableau equivalent: Overall Benchmark with Level 1 = Overall
-- ================================================================
SELECT
  `Overall`,
  MEASURE(`Total Paid Dedup Customer Payer State`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Adjustments`) AS total_adjustments,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`GPV Percent Overall LOD`) AS gpv_pct,
  MEASURE(`NPV Percent Overall LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Total Lines`) AS total_lines,
  MEASURE(`Edited Lines`) AS edited_lines,
  MEASURE(`Adjusted Lines`) AS adjusted_lines,
  MEASURE(`Adjusted Lines Percent`) AS adjusted_lines_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Original Paid per Edit`) AS original_paid_per_edit,
  MEASURE(`Original Paid per TBA Line`) AS original_paid_per_tba_line,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`Super Payer Count`) AS super_payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count,
  MEASURE(`Customer Adoption Rate`) AS customer_adoption_rate,
  MEASURE(`Payer Adoption Rate`) AS payer_adoption_rate,
  MEASURE(`GPV Concentration`) AS gpv_concentration,
  MEASURE(`NPV Concentration`) AS npv_concentration
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Overall`;


In [ ]:
%sql
-- ================================================================
-- 1. OVERALL BENCHMARK - BY CUSTOMER SEGMENT
-- Tableau Level selector dimension: Customer Segment
-- ================================================================
SELECT
  `Customer Segment`,
  MEASURE(`Total Paid Customer Segment Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by Customer Segment LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by Customer Segment LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Adjusted Lines Percent`) AS adjusted_lines_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Customer Segment`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 2. OVERALL BENCHMARK - BY LOB
-- Tableau Level selector dimension: LOB / Line of Business
-- ================================================================
SELECT
  `Line of Business`,
  MEASURE(`Total Paid LOB Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by LOB LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by LOB LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Adjusted Lines Percent`) AS adjusted_lines_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Line of Business`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 3. OVERALL BENCHMARK - BY PRODUCT TYPE
-- Tableau Level selector dimension: Product Type
-- ================================================================
SELECT
  `Product Type`,
  MEASURE(`Total Paid Product Type Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by Product Type LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by Product Type LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Product Type`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 4. OVERALL BENCHMARK - BY CLAIM TYPE
-- Tableau Level selector dimension: Claim Type
-- ================================================================
SELECT
  `Claim Type`,
  MEASURE(`Total Paid Claim Type Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by Claim Type LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by Claim Type LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Claim Type`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 5. OVERALL BENCHMARK - BY BLUES INDICATOR
-- Tableau Level selector dimension: Blues Indicator
-- ================================================================
SELECT
  `Blues Indicator`,
  MEASURE(`Total Paid Blues Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by Blues LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by Blues LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Blues Indicator`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 6. OVERALL BENCHMARK - BY STATE
-- Tableau Level selector dimension: State / Final State
-- ================================================================
SELECT
  `Final State`,
  MEASURE(`Total Paid Dedup Customer Payer State`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent Overall LOD`) AS gpv_pct,
  MEASURE(`NPV Percent Overall LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Final State`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 7. OVERALL BENCHMARK - LEVEL 1 x LEVEL 2 TEMPLATE
-- Replace dimensions with Tableau selector combinations:
--   Customer Segment, Blues Indicator, Line of Business, Product Type,
--   Claim Type, Final State, Overall
-- ================================================================
SELECT
  `Customer Segment` AS level_1,
  `Line of Business` AS level_2,
  MEASURE(`Total Paid LOB Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by LOB LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by LOB LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Customer Segment`, `Line of Business`
ORDER BY level_1, MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 8. DP BENCHMARK - DP KEY LEVEL
-- Tableau equivalent: DP Benchmark
-- Use DP-specific LOD metrics whenever Decision Point is selected.
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  MEASURE(`Total Paid DP Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Adjustments`) AS total_adjustments,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`GPV Percent by DP LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by DP LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Total Lines`) AS total_lines,
  MEASURE(`Edited Lines`) AS edited_lines,
  MEASURE(`Adjusted Lines`) AS adjusted_lines,
  MEASURE(`Adjusted Lines Percent`) AS adjusted_lines_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Original Paid per Edit`) AS original_paid_per_edit,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`Customer Adoption Rate`) AS customer_adoption_rate,
  MEASURE(`Payer Adoption Rate`) AS payer_adoption_rate,
  MEASURE(`GPV Concentration`) AS gpv_concentration,
  MEASURE(`NPV Concentration`) AS npv_concentration
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Decision Point`, `Decision Point Description`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 9. DP BENCHMARK - DP KEY x LEVEL DIMENSION
-- Example shown with LOB; replace `Line of Business` with another
-- Overall Benchmark dimension as needed.
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  `Line of Business`,
  MEASURE(`Total Paid DP Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by DP LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by DP LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Decision Point`, `Decision Point Description`, `Line of Business`
ORDER BY `Decision Point`, MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 10. DP BENCHMARK - FILTERED DP KEY
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  `Medical Policy`,
  `Topic`,
  MEASURE(`Total Paid DP Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by DP LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by DP LOD`) AS npv_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
WHERE `Decision Point` = '<dp_key>'
GROUP BY `Decision Point`, `Decision Point Description`, `Medical Policy`, `Topic`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 11. MR BENCHMARK - MID RULE LEVEL
-- Tableau equivalent: MR Benchmark
-- Use MR-specific LOD metrics whenever Mid Rule Key is selected.
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  `Mid Rule Key`,
  MEASURE(`Total Paid MR Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Adjustments`) AS total_adjustments,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`GPV Percent by MR LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by MR LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Total Lines`) AS total_lines,
  MEASURE(`Edited Lines`) AS edited_lines,
  MEASURE(`Adjusted Lines`) AS adjusted_lines,
  MEASURE(`Adjusted Lines Percent`) AS adjusted_lines_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Original Paid per Edit`) AS original_paid_per_edit,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`GPV Concentration`) AS gpv_concentration,
  MEASURE(`NPV Concentration`) AS npv_concentration
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Decision Point`, `Decision Point Description`, `Mid Rule Key`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 12. MR BENCHMARK - MID RULE x LEVEL DIMENSION
-- Example shown with Final State.
-- ================================================================
SELECT
  `Mid Rule Key`,
  `Decision Point`,
  `Final State`,
  MEASURE(`Total Paid MR Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by MR LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by MR LOD`) AS npv_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Mid Rule Key`, `Decision Point`, `Final State`
ORDER BY `Mid Rule Key`, MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 13. MR BENCHMARK - FILTERED MID RULE
-- ================================================================
SELECT
  `Mid Rule Key`,
  `Decision Point`,
  `Decision Point Description`,
  `Medical Policy`,
  `Topic`,
  MEASURE(`Total Paid MR Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by MR LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by MR LOD`) AS npv_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
WHERE `Mid Rule Key` = '<mid_rule_key>'
GROUP BY `Mid Rule Key`, `Decision Point`, `Decision Point Description`, `Medical Policy`, `Topic`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 14. POLICY-TOPIC BENCHMARK - MEDICAL POLICY LEVEL
-- Tableau equivalent: Policy-Topic Benchmark, policy grain
-- ================================================================
SELECT
  `Medical Policy`,
  MEASURE(`Total Paid Medical Policy Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Adjustments`) AS total_adjustments,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`GPV Percent by Medical Policy LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by Medical Policy LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Total Lines`) AS total_lines,
  MEASURE(`Edited Lines`) AS edited_lines,
  MEASURE(`Adjusted Lines`) AS adjusted_lines,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count,
  MEASURE(`Topic Count`) AS topic_count,
  MEASURE(`Customer Adoption Rate`) AS customer_adoption_rate,
  MEASURE(`Payer Adoption Rate`) AS payer_adoption_rate
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Medical Policy`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 15. POLICY-TOPIC BENCHMARK - TOPIC LEVEL
-- ================================================================
SELECT
  `Topic`,
  MEASURE(`Total Paid Topic Denominator`) AS paid_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Adjustments`) AS total_adjustments,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`GPV Percent by Topic LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by Topic LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count,
  MEASURE(`Medical Policy Count`) AS medical_policy_count,
  MEASURE(`Customer Adoption Rate`) AS customer_adoption_rate,
  MEASURE(`Payer Adoption Rate`) AS payer_adoption_rate
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Topic`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 16. POLICY-TOPIC BENCHMARK - MEDICAL POLICY x TOPIC
-- ================================================================
SELECT
  `Medical Policy`,
  `Topic`,
  MEASURE(`Total Paid Medical Policy Denominator`) AS paid_policy_denominator,
  MEASURE(`Total Paid Topic Denominator`) AS paid_topic_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by Medical Policy LOD`) AS gpv_pct_policy_lod,
  MEASURE(`NPV Percent by Medical Policy LOD`) AS npv_pct_policy_lod,
  MEASURE(`GPV Percent by Topic LOD`) AS gpv_pct_topic_lod,
  MEASURE(`NPV Percent by Topic LOD`) AS npv_pct_topic_lod,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Medical Policy`, `Topic`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 17. POLICY-TOPIC BENCHMARK - MEDICAL POLICY x TOPIC x DP
-- Useful for drilldown from policy/topic into DP Benchmark.
-- ================================================================
SELECT
  `Medical Policy`,
  `Topic`,
  `Decision Point`,
  `Decision Point Description`,
  MEASURE(`Total Paid DP Denominator`) AS paid_dp_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by DP LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by DP LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Medical Policy`, `Topic`, `Decision Point`, `Decision Point Description`
ORDER BY `Medical Policy`, `Topic`, MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 18. POLICY-TOPIC BENCHMARK - MEDICAL POLICY x TOPIC x DP x MR
-- Full detail drilldown grain.
-- ================================================================
SELECT
  `Medical Policy`,
  `Topic`,
  `Decision Point`,
  `Decision Point Description`,
  `Mid Rule Key`,
  MEASURE(`Total Paid MR Denominator`) AS paid_mr_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by MR LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by MR LOD`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Medical Policy`, `Topic`, `Decision Point`, `Decision Point Description`, `Mid Rule Key`
ORDER BY `Medical Policy`, `Topic`, `Decision Point`, MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 19. POLICY-TOPIC BENCHMARK - FILTERED MEDICAL POLICY
-- ================================================================
SELECT
  `Medical Policy`,
  `Topic`,
  `Decision Point`,
  `Decision Point Description`,
  MEASURE(`Total Paid Medical Policy Denominator`) AS paid_policy_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by Medical Policy LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by Medical Policy LOD`) AS npv_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
WHERE `Medical Policy` ILIKE '%<medical_policy_text>%'
GROUP BY `Medical Policy`, `Topic`, `Decision Point`, `Decision Point Description`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 20. POLICY-TOPIC BENCHMARK - FILTERED TOPIC
-- ================================================================
SELECT
  `Topic`,
  `Medical Policy`,
  `Decision Point`,
  `Decision Point Description`,
  MEASURE(`Total Paid Topic Denominator`) AS paid_topic_denominator,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent by Topic LOD`) AS gpv_pct,
  MEASURE(`NPV Percent by Topic LOD`) AS npv_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
WHERE `Topic` ILIKE '%<topic_text>%'
GROUP BY `Topic`, `Medical Policy`, `Decision Point`, `Decision Point Description`
ORDER BY MEASURE(`Total GPV`) DESC;


In [ ]:
%sql
-- ================================================================
-- 21. SOLUTION COMPARISON FOR ANY BENCHMARK LEVEL
-- This query validates PPM/CV/FWA/CPR behavior against Tableau filters.
-- ================================================================
SELECT
  `Solution Family`,
  MEASURE(`Total Paid Raw`) AS paid_raw,
  MEASURE(`Total CV Paid`) AS cv_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Percent Raw`) AS gpv_pct_raw,
  MEASURE(`NPV Percent Raw`) AS npv_pct_raw,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Solution Family`
ORDER BY `Solution Family`;


In [ ]:
%sql
-- ================================================================
-- 22. RULE/FILTER QA ACROSS BENCHMARK LEVELS
-- Useful for validating dashboard filters before exposing Genie.
-- ================================================================
SELECT
  `Rule Type`,
  `Rule Status`,
  `DP Age Bucket`,
  `Policy Collection Bundle`,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`MR Count`) AS mr_count,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics_hardened
GROUP BY `Rule Type`, `Rule Status`, `DP Age Bucket`, `Policy Collection Bundle`
ORDER BY MEASURE(`Total GPV`) DESC;
